# Reversible LM — Training at Max Sustainable Batch Size

**All-participants experiment (separate from the fixed-batch comparison).** Runs baseline and reversible (`euler`, `midpoint`) at each config's **max stable batch** and records tokens/sec, peak memory, and **final loss** at that batch.

Max batches come from the sweep in the main notebook: **baseline 56, reversible 64**. If a fresh session OOMs, drop the batch a little (64→60, 56→52) and re-run that line.

Runtime → GPU, then Run all. Results append to `results_maxbatch.jsonl` on Drive.

## 1. GPU check

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))

torch 2.11.0+cu128 | CUDA: True
GPU: Tesla T4


## 2. Dependencies

In [ ]:
!pip -q install tiktoken datasets

## 3. Mount Google Drive + working paths

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/reversible-llm-training'
DATA = f'{WORK}/data'
RESULTS = f'{WORK}/results_maxbatch.jsonl'
os.makedirs(DATA, exist_ok=True)
print('WORK =', WORK)

Mounted at /content/drive
WORK = /content/drive/MyDrive/reversible-llm-training


## 4. Write the code files

In [ ]:
%%writefile model.py
"""
Reversible language model — baseline + three reversible integrators.

Method follows Karami et al., "Reversing Large Language Models for Efficient
Training and Fine-Tuning" (arXiv:2512.02056): reversible LLM layers cast as
time-reversible / symplectic ODE integrators, so hidden states are reconstructed
during backprop instead of stored.

Configs (select with `coupling`):
  "none"     : standard pre-norm Transformer (stores activations)   -> baseline
  "euler"    : Hamiltonian coupled-stream additive coupling (1st order)
                 q = q_prev + Attn(LN1(p_prev))
                 p = p_prev + MLP (LN2(q))
  "midpoint" : explicit-midpoint / leapfrog multi-step integrator
                 p^{l+1} = p^{l-1} + step * f(p^{l})
  "leapfrog" : Stormer-Verlet 2nd-order integrator
                 p^{l+1} = 2 p^{l} - p^{l-1} + step * f(p^{l})

All reversible blocks carry TWO full-width streams (u, v) and reuse the same
Attn + MLP as the baseline, so every config has the SAME parameter count.

NOTE: exact reconstruction requires the layer function to be deterministic, so
reversible modes run with dropout = 0 (enforced below).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Sublayers
# ----------------------------------------------------------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.0):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_model = d_model
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.d_model, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class MLP(nn.Module):
    def __init__(self, d_model, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.fc = nn.Linear(d_model, mlp_ratio * d_model)
        self.proj = nn.Linear(mlp_ratio * d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))


class SubLayer(nn.Module):
    """LayerNorm -> fn (pre-norm)."""
    def __init__(self, d_model, fn):
        super().__init__()
        self.ln = nn.LayerNorm(d_model)
        self.fn = fn

    def forward(self, x):
        return self.fn(self.ln(x))


class LayerFn(nn.Module):
    """Full layer step f(p) = Attn(LN1(p)) + MLP(LN2(p + Attn(...)));
    the residual 'delta' of a standard pre-norm block. Used by the single-stream
    integrators (midpoint, leapfrog). Same params as a baseline block."""
    def __init__(self, d_model, n_head, mlp_ratio, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, mlp_ratio, dropout)

    def forward(self, p):
        a = self.attn(self.ln1(p))
        m = self.mlp(self.ln2(p + a))
        return a + m


# ----------------------------------------------------------------------------
# Baseline (non-reversible) block
# ----------------------------------------------------------------------------
class StandardBlock(nn.Module):
    def __init__(self, d_model, n_head, mlp_ratio, dropout):
        super().__init__()
        self.attn = SubLayer(d_model, CausalSelfAttention(d_model, n_head, dropout))
        self.ffn = SubLayer(d_model, MLP(d_model, mlp_ratio, dropout))

    def forward(self, x):
        x = x + self.attn(x)
        x = x + self.ffn(x)
        return x


# ----------------------------------------------------------------------------
# Reversible blocks: each defines forward(u,v) and exact inverse(u',v').
# The memory-saving backward is shared in the base class.
# ----------------------------------------------------------------------------
class ReversibleBase(nn.Module):
    def backward_pass(self, u2, v2, du2, dv2):
        """Reconstruct inputs from outputs, recompute forward with grad,
        backprop for input grads + accumulate param grads."""
        with torch.no_grad():
            u1, v1 = self.inverse(u2, v2)
        with torch.enable_grad():
            u1 = u1.detach().requires_grad_(True)
            v1 = v1.detach().requires_grad_(True)
            uo, vo = self.forward(u1, v1)
            torch.autograd.backward((uo, vo), (du2, dv2))
        with torch.no_grad():
            du1, dv1 = u1.grad, v1.grad
            u1, v1 = u1.detach(), v1.detach()
        return u1, v1, du1, dv1


class HamiltonianBlock(ReversibleBase):
    """euler: q = q_prev + F(p_prev);  p = p_prev + G(q).
    Here u=q (momentum), v=p (position). F=Attn, G=MLP."""
    def __init__(self, d_model, n_head, mlp_ratio, dropout):
        super().__init__()
        self.F = SubLayer(d_model, CausalSelfAttention(d_model, n_head, dropout))
        self.G = SubLayer(d_model, MLP(d_model, mlp_ratio, dropout))

    def forward(self, u, v):          # (q_prev, p_prev)
        q = u + self.F(v)
        p = v + self.G(q)
        return q, p

    def inverse(self, u2, v2):        # (q, p)
        p_prev = v2 - self.G(u2)
        q_prev = u2 - self.F(p_prev)
        return q_prev, p_prev


class MidpointBlock(ReversibleBase):
    """Explicit midpoint (paper eq.):  p^{l+1} = p^{l-1} + 2h * f(p^{l}).
    State (u,v) = (p_prev, p_cur). The block is evaluated at the INTERMEDIATE
    state p_cur, which the backward pass already holds — that is what makes the
    reverse step  p^{l-1} = p^{l+1} - 2h * f(p^{l})  exact without inverting f.
    `h` is the fixed step size; stable only in a narrow range (see paper)."""
    def __init__(self, d_model, n_head, mlp_ratio, dropout, h=0.5):
        super().__init__()
        self.f = LayerFn(d_model, n_head, mlp_ratio, dropout)
        self.c = 2.0 * h                          # coefficient 2h

    def forward(self, u, v):          # (p_prev, p_cur) -> (p_cur, p_next)
        p_next = u + self.c * self.f(v)
        return v, p_next

    def inverse(self, u2, v2):        # (p_cur, p_next) -> (p_prev, p_cur)
        p_prev = v2 - self.c * self.f(u2)
        return p_prev, u2


class LeapfrogBlock(ReversibleBase):
    """Stormer-Verlet 2nd-order:  p^{l+1} = 2 p^{l} - p^{l-1} + h^2 * f(p^{l}).
    State (u,v) = (p_prev, p_cur)."""
    def __init__(self, d_model, n_head, mlp_ratio, dropout, h=0.5):
        super().__init__()
        self.f = LayerFn(d_model, n_head, mlp_ratio, dropout)
        self.c = h * h                            # coefficient h^2

    def forward(self, u, v):          # (p_prev, p_cur) -> (p_cur, p_next)
        p_next = 2 * v - u + self.c * self.f(v)
        return v, p_next

    def inverse(self, u2, v2):        # (p_cur, p_next) -> (p_prev, p_cur)
        p_prev = 2 * u2 - v2 + self.c * self.f(u2)
        return p_prev, u2


def make_reversible_block(coupling, d_model, n_head, mlp_ratio, dropout, h):
    if coupling == "euler":
        return HamiltonianBlock(d_model, n_head, mlp_ratio, dropout)
    if coupling == "midpoint":
        return MidpointBlock(d_model, n_head, mlp_ratio, dropout, h)
    if coupling == "leapfrog":
        return LeapfrogBlock(d_model, n_head, mlp_ratio, dropout, h)
    raise ValueError(coupling)


# ----------------------------------------------------------------------------
# Memory-saving reversible stack
# ----------------------------------------------------------------------------
class _ReversibleFunction(torch.autograd.Function):
    """Runs a stack of reversible blocks storing ONLY the final output;
    activations are reconstructed block-by-block in backward."""
    @staticmethod
    def forward(ctx, x, blocks):
        ctx.blocks = blocks
        u, v = torch.chunk(x, 2, dim=-1)
        with torch.no_grad():
            for blk in blocks:
                u, v = blk(u, v)
        out = torch.cat([u, v], dim=-1)
        ctx.save_for_backward(out)
        return out

    @staticmethod
    def backward(ctx, grad_out):
        out, = ctx.saved_tensors
        u, v = torch.chunk(out, 2, dim=-1)
        du, dv = torch.chunk(grad_out, 2, dim=-1)
        for blk in reversed(ctx.blocks):
            u, v, du, dv = blk.backward_pass(u, v, du, dv)
        return torch.cat([du, dv], dim=-1), None


class ReversibleStack(nn.Module):
    def __init__(self, blocks):
        super().__init__()
        self.blocks = nn.ModuleList(blocks)

    def forward(self, x):
        # seed the two coupled streams from the embedding
        x = torch.cat([x, x], dim=-1)
        x = _ReversibleFunction.apply(x, self.blocks)
        u, v = torch.chunk(x, 2, dim=-1)
        return v  # the "position" stream


# ----------------------------------------------------------------------------
# Full GPT
# ----------------------------------------------------------------------------
class GPTConfig:
    def __init__(self, vocab_size=50257, seq_len=256, n_layer=10, d_model=256,
                 n_head=8, mlp_ratio=4, dropout=0.0, coupling="none", h=0.5):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.n_layer = n_layer
        self.d_model = d_model
        self.n_head = n_head
        self.mlp_ratio = mlp_ratio
        self.dropout = dropout
        self.coupling = coupling            # none | euler | midpoint | leapfrog
        self.h = h                          # integrator step size (midpoint uses 2h, leapfrog h^2)


class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        drop = cfg.dropout
        if cfg.coupling != "none" and drop != 0.0:
            print("[model] reversible mode requires deterministic layers; forcing dropout=0")
            drop = 0.0

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.seq_len, cfg.d_model)
        self.drop = nn.Dropout(drop)

        if cfg.coupling == "none":
            self.stack = nn.ModuleList([
                StandardBlock(cfg.d_model, cfg.n_head, cfg.mlp_ratio, drop)
                for _ in range(cfg.n_layer)
            ])
            self.reversible = False
        else:
            self.stack = ReversibleStack([
                make_reversible_block(cfg.coupling, cfg.d_model, cfg.n_head,
                                      cfg.mlp_ratio, drop, cfg.h)
                for _ in range(cfg.n_layer)
            ])
            self.reversible = True

        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight  # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        n = sum(p.numel() for p in self.parameters())
        return n - self.pos_emb.weight.numel()

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        if self.reversible:
            x = self.stack(x)
        else:
            for blk in self.stack:
                x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-1)
        return logits, loss

    def configure_optimizer(self, weight_decay, lr, betas, device_type):
        decay, no_decay = [], []
        for p in self.parameters():
            if not p.requires_grad:
                continue
            (decay if p.dim() >= 2 else no_decay).append(p)
        groups = [
            {"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ]
        return torch.optim.AdamW(groups, lr=lr, betas=betas,
                                 fused=(device_type == "cuda"))


Writing model.py


In [ ]:
%%writefile data.py
"""
Prepare ~50M tokens of training data, nanoGPT-style.

Tokenizes text with the GPT-2 BPE (tiktoken) and writes uint16 binaries:
  <out_dir>/train.bin   (~95%)
  <out_dir>/val.bin     (~5%)
  <out_dir>/meta.json   (vocab_size, token counts)

Two sources:
  --source hf     a HuggingFace dataset streamed until ~N tokens (default)
  --source file   a local .txt file you point at with --path

Examples
--------
  # default: TinyStories, ~50M tokens
  python data.py --tokens 50_000_000

  # wikitext-103 raw
  python data.py --dataset wikitext --config wikitext-103-raw-v1 --tokens 50_000_000

  # your own text file
  python data.py --source file --path corpus.txt --tokens 50_000_000
"""
import os
import json
import argparse
import numpy as np


def get_encoder():
    import tiktoken
    return tiktoken.get_encoding("gpt2")


def iter_texts_hf(dataset, config, split, text_field):
    from datasets import load_dataset
    ds = load_dataset(dataset, config, split=split, streaming=True)
    for ex in ds:
        t = ex.get(text_field)
        if t:
            yield t


def iter_texts_file(path):
    # stream a big text file in ~1MB chunks split on blank lines
    buf = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            buf.append(line)
            if sum(len(b) for b in buf) > 1_000_000:
                yield "".join(buf)
                buf = []
    if buf:
        yield "".join(buf)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--source", choices=["hf", "file"], default="hf")
    ap.add_argument("--dataset", default="roneneldan/TinyStories")
    ap.add_argument("--config", default=None)
    ap.add_argument("--split", default="train")
    ap.add_argument("--text_field", default="text")
    ap.add_argument("--path", default=None, help="local .txt (for --source file)")
    ap.add_argument("--tokens", type=int, default=50_000_000)
    ap.add_argument("--val_frac", type=float, default=0.05)
    ap.add_argument("--out_dir", default="data")
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    enc = get_encoder()
    eot = enc.eot_token

    if args.source == "file":
        assert args.path, "--path required for --source file"
        texts = iter_texts_file(args.path)
    else:
        texts = iter_texts_hf(args.dataset, args.config, args.split, args.text_field)

    print(f"Tokenizing up to {args.tokens:,} tokens with GPT-2 BPE ...")
    ids = np.empty(args.tokens + 4096, dtype=np.uint16)
    n = 0
    for i, text in enumerate(texts):
        toks = enc.encode_ordinary(text)
        toks.append(eot)
        k = min(len(toks), len(ids) - n)
        ids[n:n + k] = np.array(toks[:k], dtype=np.uint16)
        n += k
        if i % 2000 == 0:
            print(f"  {n:,} tokens", end="\r")
        if n >= args.tokens:
            break
    ids = ids[:n]
    print(f"\nCollected {n:,} tokens.")

    n_val = int(n * args.val_frac)
    val = ids[:n_val]
    train = ids[n_val:]
    train.tofile(os.path.join(args.out_dir, "train.bin"))
    val.tofile(os.path.join(args.out_dir, "val.bin"))

    meta = {"vocab_size": enc.n_vocab, "encoding": "gpt2",
            "train_tokens": int(train.size), "val_tokens": int(val.size)}
    with open(os.path.join(args.out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print(f"Wrote train.bin ({train.size:,}), val.bin ({val.size:,}) to {args.out_dir}/")
    print(f"meta: {meta}")


if __name__ == "__main__":
    main()


Writing data.py


In [ ]:
%%writefile train.py
"""
Train the 20M LM. One script, three configs via --coupling {none,euler,midpoint}.

Logs the four metrics the assignment asks for and appends one JSON line to
results.jsonl per run:
    coupling, batch_size, seq_len, params, final_train_loss, val_loss,
    tokens_per_sec, peak_mem_gb, max_batch (if using the sweep).

Keep seq_len, max_iters and the data identical across the three runs so the
comparison is clean — only change --coupling and --batch_size.

Examples
--------
  python train.py --coupling none     --batch_size 32 --max_iters 3000
  python train.py --coupling euler    --batch_size 32 --max_iters 3000
  python train.py --coupling midpoint --batch_size 32 --max_iters 3000
"""
import os
import json
import time
import math
import argparse
import numpy as np
import torch
from model import GPT, GPTConfig


def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i + block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + 1 + block_size].astype(np.int64)) for i in ix])
    if device.startswith("cuda"):
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


def lr_at(it, warmup, max_iters, lr, min_lr):
    if it < warmup:
        return lr * (it + 1) / warmup
    if it > max_iters:
        return min_lr
    ratio = (it - warmup) / max(1, (max_iters - warmup))
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return min_lr + coeff * (lr - min_lr)


@torch.no_grad()
def estimate_val_loss(model, val_data, block_size, batch_size, device, iters=50):
    model.eval()
    losses = []
    for _ in range(iters):
        x, y = get_batch(val_data, block_size, batch_size, device)
        with torch.autocast(device_type="cuda" if device.startswith("cuda") else "cpu",
                            dtype=torch.bfloat16, enabled=device.startswith("cuda")):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def build_model(args, vocab_size, device):
    cfg = GPTConfig(vocab_size=vocab_size, seq_len=args.seq_len,
                    n_layer=args.n_layer, d_model=args.d_model,
                    n_head=args.n_head, mlp_ratio=4, dropout=args.dropout,
                    coupling=args.coupling, h=args.h)
    model = GPT(cfg).to(device)
    if args.compile and hasattr(torch, "compile"):
        model = torch.compile(model)
    return model, cfg


def run(args):
    device = args.device
    torch.manual_seed(args.seed)
    if device.startswith("cuda"):
        torch.cuda.manual_seed(args.seed)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    meta_path = os.path.join(args.data_dir, "meta.json")
    vocab_size = 50257
    if os.path.exists(meta_path):
        vocab_size = json.load(open(meta_path))["vocab_size"]
    train_data = np.memmap(os.path.join(args.data_dir, "train.bin"), dtype=np.uint16, mode="r")
    val_data = np.memmap(os.path.join(args.data_dir, "val.bin"), dtype=np.uint16, mode="r")

    model, cfg = build_model(args, vocab_size, device)
    raw = model._orig_mod if hasattr(model, "_orig_mod") else model
    n_params = raw.num_params()
    print(f"coupling={args.coupling} | params={n_params/1e6:.2f}M | "
          f"batch={args.batch_size} seq={args.seq_len} device={device}")

    opt = raw.configure_optimizer(args.weight_decay, args.lr, (0.9, 0.95),
                                  "cuda" if device.startswith("cuda") else "cpu")
    use_amp = device.startswith("cuda")

    if device.startswith("cuda"):
        torch.cuda.reset_peak_memory_stats()

    model.train()
    t0 = time.time()
    tokens_seen = 0
    last_loss = float("nan")
    for it in range(args.max_iters):
        lr = lr_at(it, args.warmup, args.max_iters, args.lr, args.lr / 10)
        for g in opt.param_groups:
            g["lr"] = lr

        x, y = get_batch(train_data, args.seq_len, args.batch_size, device)
        with torch.autocast(device_type="cuda" if use_amp else "cpu",
                            dtype=torch.bfloat16, enabled=use_amp):
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        tokens_seen += args.batch_size * args.seq_len
        last_loss = loss.item()
        if it % args.log_every == 0 or it == args.max_iters - 1:
            if device.startswith("cuda"):
                torch.cuda.synchronize()
            elapsed = time.time() - t0
            tps = tokens_seen / max(elapsed, 1e-9)
            print(f"  it {it:5d} | loss {last_loss:.4f} | lr {lr:.2e} | {tps:,.0f} tok/s")

    if device.startswith("cuda"):
        torch.cuda.synchronize()
    elapsed = time.time() - t0
    tokens_per_sec = tokens_seen / max(elapsed, 1e-9)
    peak_mem_gb = (torch.cuda.max_memory_allocated() / 1e9) if device.startswith("cuda") else 0.0
    val_loss = estimate_val_loss(model, val_data, args.seq_len, args.batch_size, device) \
        if args.eval_val else None

    result = {
        "coupling": args.coupling,
        "batch_size": args.batch_size,
        "seq_len": args.seq_len,
        "n_layer": args.n_layer,
        "d_model": args.d_model,
        "params_millions": round(n_params / 1e6, 3),
        "max_iters": args.max_iters,
        "final_train_loss": round(last_loss, 4),
        "val_loss": round(val_loss, 4) if val_loss is not None else None,
        "tokens_per_sec": round(tokens_per_sec, 1),
        "peak_mem_gb": round(peak_mem_gb, 3),
    }
    print("RESULT " + json.dumps(result))
    with open(args.results, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def build_argparser():
    ap = argparse.ArgumentParser()
    ap.add_argument("--coupling", choices=["none", "euler", "midpoint", "leapfrog"], default="none")
    ap.add_argument("--h", type=float, default=0.5, help="integrator step size (midpoint uses 2h, leapfrog h^2)")
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--seq_len", type=int, default=256)
    ap.add_argument("--n_layer", type=int, default=10)
    ap.add_argument("--d_model", type=int, default=256)
    ap.add_argument("--n_head", type=int, default=8)
    ap.add_argument("--dropout", type=float, default=0.0)
    ap.add_argument("--max_iters", type=int, default=3000)
    ap.add_argument("--warmup", type=int, default=100)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--weight_decay", type=float, default=0.1)
    ap.add_argument("--data_dir", default="data")
    ap.add_argument("--results", default="results.jsonl")
    ap.add_argument("--log_every", type=int, default=100)
    ap.add_argument("--eval_val", action="store_true")
    ap.add_argument("--compile", action="store_true")
    ap.add_argument("--seed", type=int, default=1337)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    return ap


if __name__ == "__main__":
    run(build_argparser().parse_args())


Writing train.py


In [ ]:
%%writefile sweep.py
"""
Find the maximum stable batch size for a coupling, then report throughput/memory
at that batch. Doubles the batch until CUDA OOM, then does one linear back-off
step. Runs a short burn-in (few iters) at each candidate — fast.

Usage:
  python sweep.py --coupling none
  python sweep.py --coupling euler
  python sweep.py --coupling midpoint

Appends the winning config to results.jsonl (tagged with "max_batch": true) and
prints a one-line summary you can paste straight into the report.
"""
import json
import argparse
import torch
from train import build_argparser, run


def try_batch(base_args, batch):
    args = argparse.Namespace(**vars(base_args))
    args.batch_size = batch
    args.max_iters = base_args.probe_iters
    args.eval_val = False
    args.results = "/dev/null"  # don't log probe runs
    try:
        res = run(args)
        torch.cuda.empty_cache()
        return res
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return None
        raise


def main():
    ap = build_argparser()
    ap.add_argument("--start_batch", type=int, default=8)
    ap.add_argument("--probe_iters", type=int, default=60)
    args = ap.parse_args()

    if not args.device.startswith("cuda"):
        print("WARNING: OOM sweep is only meaningful on CUDA. Running anyway for smoke test.")

    best = None
    batch = args.start_batch
    last_ok = None
    print(f"=== batch-size sweep: coupling={args.coupling} ===")
    while True:
        print(f"-- trying batch_size={batch}")
        res = try_batch(args, batch)
        if res is None:
            print(f"   OOM at batch_size={batch}")
            break
        last_ok = res
        best = batch
        batch *= 2
        if batch > 100000:
            break

    if best is None:
        print("Could not fit even the starting batch size; lower --start_batch.")
        return

    # linear back-off: try to squeeze between best and 2*best
    hi = best * 2
    probe = best + (hi - best) // 2
    while probe > best:
        print(f"-- back-off probe batch_size={probe}")
        res = try_batch(args, probe)
        if res is not None:
            best = probe
            last_ok = res
            probe = best + (hi - best) // 2
        else:
            hi = probe
            probe = best + (hi - best) // 2

    summary = {
        "coupling": args.coupling,
        "max_batch": best,
        "seq_len": args.seq_len,
        "params_millions": last_ok["params_millions"],
        "tokens_per_sec_at_max": last_ok["tokens_per_sec"],
        "peak_mem_gb_at_max": last_ok["peak_mem_gb"],
    }
    print("MAXBATCH " + json.dumps(summary))
    with open(args.results, "a") as f:
        f.write(json.dumps({**summary, "is_max_batch_sweep": True}) + "\n")
    print(f"\n>> {args.coupling}: max stable batch = {best}, "
          f"{last_ok['tokens_per_sec']:,.0f} tok/s, {last_ok['peak_mem_gb']:.2f} GB peak")


if __name__ == "__main__":
    main()


Writing sweep.py


In [ ]:
%%writefile test_reversible.py
"""
Correctness checks for the reversible blocks. Run on CPU:  python test_reversible.py

1. Exact invertibility:      inverse(forward(x)) == x
2. Gradient equivalence:     custom memory-saving backward == plain autograd
   (a reference model stores activations; a reversible model recomputes them;
    param grads and input grads must match to numerical tolerance)
"""
import torch
import torch.nn as nn
from model import make_reversible_block, _ReversibleFunction, GPT, GPTConfig

torch.manual_seed(0)
COUPLINGS = ("euler", "midpoint", "leapfrog")


def test_invertibility():
    print("== invertibility ==")
    B, T, d = 2, 8, 32
    for coupling in COUPLINGS:
        blk = make_reversible_block(coupling, d, 4, 4, 0.0, 0.5).eval()
        u = torch.randn(B, T, d)
        v = torch.randn(B, T, d)
        with torch.no_grad():
            y1, y2 = blk(u, v)
            ru, rv = blk.inverse(y1, y2)
        err = (ru - u).abs().max().item() + (rv - v).abs().max().item()
        print(f"  {coupling:9s} reconstruction max-err = {err:.2e}", "OK" if err < 1e-4 else "FAIL")
        assert err < 1e-4


def test_grad_equivalence():
    print("== gradient equivalence (reversible backward vs plain autograd) ==")
    B, T, d = 2, 8, 32
    for coupling in COUPLINGS:
        blocks = nn.ModuleList([
            make_reversible_block(coupling, d, 4, 4, 0.0, 0.5) for _ in range(3)
        ]).eval()

        x = torch.randn(B, T, 2 * d, requires_grad=True)

        # --- reference: plain autograd through the same blocks ---
        x_ref = x.detach().clone().requires_grad_(True)
        a1, a2 = torch.chunk(x_ref, 2, dim=-1)
        for blk in blocks:
            a1, a2 = blk(a1, a2)
        out_ref = torch.cat([a1, a2], dim=-1)
        loss_ref = out_ref.pow(2).mean()
        ref_grads = torch.autograd.grad(loss_ref, [x_ref] + list(blocks.parameters()),
                                        retain_graph=False)
        ref_dx = ref_grads[0]
        ref_pg = ref_grads[1:]

        # --- reversible: custom Function, activations recomputed ---
        for p in blocks.parameters():
            p.grad = None
        out = _ReversibleFunction.apply(x, blocks)
        loss = out.pow(2).mean()
        loss.backward()
        rev_dx = x.grad
        rev_pg = [p.grad for p in blocks.parameters()]

        dx_err = (rev_dx - ref_dx).abs().max().item()
        pg_err = max((a - b).abs().max().item() for a, b in zip(rev_pg, ref_pg))
        print(f"  {coupling:9s} input-grad err = {dx_err:.2e} | param-grad err = {pg_err:.2e}",
              "OK" if max(dx_err, pg_err) < 1e-4 else "FAIL")
        assert max(dx_err, pg_err) < 1e-4


def test_param_parity():
    print("== parameter-count parity across all configs ==")
    counts = {}
    for coupling in ("none",) + COUPLINGS:
        cfg = GPTConfig(vocab_size=50257, seq_len=128, n_layer=6,
                        d_model=384, n_head=6, coupling=coupling)
        counts[coupling] = GPT(cfg).num_params()
    for k, v in counts.items():
        print(f"  {k:9s} params = {v/1e6:.2f}M")
    assert len(set(counts.values())) == 1
    print("  all identical  OK")


def test_no_activation_storage():
    """A reversible forward under no_grad must not build an autograd graph over
    intermediate activations — only the final output carries grad."""
    print("== reversible forward stores no intermediate graph ==")
    B, T, d = 2, 8, 16
    blocks = nn.ModuleList([make_reversible_block("midpoint", d, 4, 4, 0.0, 0.5) for _ in range(4)])
    x = torch.randn(B, T, 2 * d, requires_grad=True)
    out = _ReversibleFunction.apply(x, blocks)
    # grad_fn is the custom Function; no per-layer graph retained
    ok = out.grad_fn is not None and "ReversibleFunctionBackward" in type(out.grad_fn).__name__
    print(f"  output backed by custom reversible Function: {ok}", "OK" if ok else "FAIL")
    assert ok


if __name__ == "__main__":
    test_invertibility()
    test_grad_equivalence()
    test_param_parity()
    test_no_activation_storage()
    print("\nAll checks passed.")


Writing test_reversible.py


## 5. Data — tokenize ~50M tokens to Drive (skips if present)

In [ ]:
if os.path.exists(f'{DATA}/train.bin'):
    print('Data already on Drive — skipping.')
else:
    !python data.py --tokens 50_000_000 --out_dir "{DATA}"

Data already on Drive — skipping.


## 6. Full training at max sustainable batch — one cell per config

In [ ]:
if os.path.exists(RESULTS): os.remove(RESULTS)
print('results reset')

results reset


In [ ]:
!python train.py --coupling none     --batch_size 56 --max_iters 2000 --eval_val --data_dir "{DATA}" --results "{RESULTS}"

coupling=none | params=20.75M | batch=56 seq=256 device=cuda
  it     0 | loss 10.8721 | lr 3.00e-06 | 8,794 tok/s
  it   100 | loss 5.7247 | lr 3.00e-04 | 13,069 tok/s
  it   200 | loss 4.1603 | lr 2.98e-04 | 12,888 tok/s
  it   300 | loss 3.8153 | lr 2.93e-04 | 12,808 tok/s
  it   400 | loss 3.5676 | lr 2.84e-04 | 12,764 tok/s
  it   500 | loss 3.2846 | lr 2.72e-04 | 12,733 tok/s
  it   600 | loss 3.1469 | lr 2.56e-04 | 12,712 tok/s
  it   700 | loss 3.0104 | lr 2.39e-04 | 12,700 tok/s
  it   800 | loss 3.0027 | lr 2.19e-04 | 12,690 tok/s
  it   900 | loss 3.0105 | lr 1.98e-04 | 12,684 tok/s
  it  1000 | loss 2.7151 | lr 1.76e-04 | 12,678 tok/s
  it  1100 | loss 2.8122 | lr 1.54e-04 | 12,675 tok/s
  it  1200 | loss 2.6803 | lr 1.32e-04 | 12,673 tok/s
  it  1300 | loss 2.7379 | lr 1.11e-04 | 12,670 tok/s
  it  1400 | loss 2.7295 | lr 9.12e-05 | 12,667 tok/s
  it  1500 | loss 2.5358 | lr 7.36e-05 | 12,664 tok/s
  it  1600 | loss 2.6467 | lr 5.85e-05 | 12,662 tok/s
  it  1700 | loss 2.6

In [ ]:
!python train.py --coupling euler    --batch_size 64 --max_iters 2000 --eval_val --data_dir "{DATA}" --results "{RESULTS}"

coupling=euler | params=20.75M | batch=64 seq=256 device=cuda
  it     0 | loss 10.9107 | lr 3.00e-06 | 9,925 tok/s
  it   100 | loss 5.8408 | lr 3.00e-04 | 11,643 tok/s
  it   200 | loss 4.3151 | lr 2.98e-04 | 11,603 tok/s
  it   300 | loss 3.8211 | lr 2.93e-04 | 11,581 tok/s
  it   400 | loss 3.5507 | lr 2.84e-04 | 11,571 tok/s
  it   500 | loss 3.3673 | lr 2.72e-04 | 11,559 tok/s
  it   600 | loss 3.1888 | lr 2.56e-04 | 11,550 tok/s
  it   700 | loss 3.1172 | lr 2.39e-04 | 11,546 tok/s
  it   800 | loss 2.9591 | lr 2.19e-04 | 11,542 tok/s
  it   900 | loss 2.8523 | lr 1.98e-04 | 11,538 tok/s
  it  1000 | loss 2.7728 | lr 1.76e-04 | 11,535 tok/s
  it  1100 | loss 2.7774 | lr 1.54e-04 | 11,531 tok/s
  it  1200 | loss 2.6971 | lr 1.32e-04 | 11,529 tok/s
  it  1300 | loss 2.6021 | lr 1.11e-04 | 11,527 tok/s
  it  1400 | loss 2.6581 | lr 9.12e-05 | 11,522 tok/s
  it  1500 | loss 2.5659 | lr 7.36e-05 | 11,519 tok/s
  it  1600 | loss 2.5254 | lr 5.85e-05 | 11,520 tok/s
  it  1700 | loss 2.

In [ ]:
!ls *.py

data.py  model.py  sweep.py  test_reversible.py  train.py


In [17]:
!python train.py --coupling midpoint --batch_size 64 --max_iters 2000 --eval_val --data_dir "{DATA}" --results "{RESULTS}"

coupling=midpoint | params=20.75M | batch=64 seq=256 device=cuda
  it     0 | loss 10.8931 | lr 3.00e-06 | 751 tok/s
  it   100 | loss 5.6829 | lr 3.00e-04 | 10,910 tok/s
  it   200 | loss 4.2152 | lr 2.98e-04 | 11,825 tok/s
  it   300 | loss 3.7308 | lr 2.93e-04 | 12,015 tok/s
  it   400 | loss 3.4365 | lr 2.84e-04 | 12,076 tok/s
  it   500 | loss 3.2355 | lr 2.72e-04 | 12,104 tok/s
  it   600 | loss 3.0762 | lr 2.56e-04 | 12,124 tok/s
  it   700 | loss 3.0247 | lr 2.39e-04 | 12,135 tok/s
  it   800 | loss 2.8628 | lr 2.19e-04 | 12,142 tok/s
  it   900 | loss 2.7694 | lr 1.98e-04 | 12,162 tok/s
  it  1000 | loss 2.7025 | lr 1.76e-04 | 12,185 tok/s
  it  1100 | loss 2.7077 | lr 1.54e-04 | 12,203 tok/s
  it  1200 | loss 2.6301 | lr 1.32e-04 | 12,217 tok/s
  it  1300 | loss 2.5314 | lr 1.11e-04 | 12,229 tok/s
  it  1400 | loss 2.5866 | lr 9.12e-05 | 12,240 tok/s
  it  1500 | loss 2.4885 | lr 7.36e-05 | 12,250 tok/s
  it  1600 | loss 2.4628 | lr 5.85e-05 | 12,255 tok/s
  it  1700 | loss 2

## 7. Results → CSV on Drive

In [18]:
import json, pandas as pd
rows=[json.loads(l) for l in open(RESULTS)] if os.path.exists(RESULTS) else []
df=pd.DataFrame(rows); pd.set_option('display.max_columns',None); display(df)
df.to_csv(f'{WORK}/results_maxbatch.csv', index=False); print('saved results_maxbatch.csv')

,coupling,batch_size,seq_len,n_layer,d_model,params_millions,max_iters,final_train_loss,val_loss,tokens_per_sec,peak_mem_gb
0,none,56,256,10,256,20.754,2000,2.5358,2.5715,12650.9,11.698
1,euler,64,256,10,256,20.754,2000,2.4885,2.5471,11517.0,10.221
2,midpoint,64,256,10,256,20.754,2000,2.4109,2.4753,12261.0,10.221


saved results_maxbatch.csv
